# ⚡ iFixit RepairRAG — Hybrid Pipeline + Impressive UI

**BM25 + BGE Dense → RRF → Cross-Encoder → GPT-4o**

### Instructions
1. Run **cell 1** — installs all dependencies
2. Run **cell 2** — mounts Google Drive
3. **Cell 3** — paste your ngrok token (free at dashboard.ngrok.com) and run
4. Run **cell 4** — writes the app to disk
5. Run **cell 5** — launches Streamlit
6. Run **cell 6** — opens ngrok tunnel → click the blue button
7. Paste your **OpenAI API key** in the app sidebar

### Files used from Drive
```
MyDrive/ifixit_dump/final_hybrid_rag/
    ├── index.faiss   (646 MB)
    ├── meta.jsonl    (408 MB)
    └── bm25.pkl      (218 MB)
```

In [37]:
# 1. Install dependencies
!pip install -q streamlit faiss-cpu sentence-transformers openai rank-bm25 numpy

In [38]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
# 1. Install dependencies
!pip install -q streamlit faiss-cpu sentence-transformers openai rank-bm25 numpy pyngrok
!pyngrok install-ngrok  # installs the ngrok binary

ngrok - tunnel local ports to public URLs and inspect traffic

USAGE:
  ngrok [command] [flags]

COMMANDS: 
  api             CLI to api.ngrok.com
  completion      generates shell completion code for bash or zsh
  config          update or migrate ngrok's configuration file
  credits         prints author and licensing information
  help            help about any command
  http            start an HTTP tunnel
  service         run and control ngrok as a background service
  start           start endpoints in the config file by name
  tcp             start a TCP tunnel
  tls             start a TLS endpoint
  update          update ngrok to the latest version
  version         print the version string

EXAMPLES: 
# forward http traffic from assigned public URL to local port 80
ngrok http 80
# port 8080 available at baz.ngrok.dev
ngrok http --url baz.ngrok.dev 8080
# tunnel arbitrary TCP traffic to port 22
ngrok tcp 22
# secure your app with oauth
ngrok http 80 --oauth=google --oauth-al

In [40]:
# 3. Set your ngrok auth token
# Free token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "3CKVeP1rhT27xBnfStTgZPVnxBs_3Dfzgco3gvWQ1TCdVJewf"  # <-- paste here

!ngrok config add-authtoken {NGROK_TOKEN}
print('✓ ngrok token configured')

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
✓ ngrok token configured


In [41]:
%%writefile /content/app.py
import json, time, pickle, os, re
from pathlib import Path
import faiss, numpy as np, streamlit as st
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

st.set_page_config(page_title="RepairRAG", page_icon="\U0001f527", layout="wide")

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Rajdhani:wght@400;500;600;700&family=Exo+2:wght@300;400;600;700&display=swap');

*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
#MainMenu, footer, header, [data-testid="stToolbar"] { visibility: hidden; }

/* ── Root ── */
.stApp {
    background: #050810;
    background-image:
        radial-gradient(ellipse 80% 50% at 50% -10%, rgba(0,200,255,0.07) 0%, transparent 60%),
        radial-gradient(ellipse 40% 30% at 90% 90%, rgba(255,100,0,0.05) 0%, transparent 50%);
    font-family: 'Exo 2', sans-serif;
}

section[data-testid="stMain"] .block-container {
    padding: 1.5rem 2rem 3rem 2rem;
    max-width: 1200px;
}

/* ── Circuit board background SVG overlay ── */
.circuit-bg {
    position: fixed; top: 0; left: 0; width: 100%; height: 100%;
    pointer-events: none; z-index: 0; opacity: 0.03;
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='100' height='100'%3E%3Cpath d='M10 10h80M10 50h30M60 50h30M50 10v30M50 60v30M10 90h30M60 90h30' stroke='%2300cfff' stroke-width='1' fill='none'/%3E%3Ccircle cx='50' cy='50' r='4' fill='%2300cfff'/%3E%3Ccircle cx='10' cy='10' r='3' fill='%2300cfff'/%3E%3Ccircle cx='90' cy='10' r='3' fill='%2300cfff'/%3E%3Ccircle cx='10' cy='90' r='3' fill='%2300cfff'/%3E%3Ccircle cx='90' cy='90' r='3' fill='%2300cfff'/%3E%3Ccircle cx='50' cy='10' r='2' fill='%2300cfff'/%3E%3Ccircle cx='50' cy='90' r='2' fill='%2300cfff'/%3E%3Ccircle cx='10' cy='50' r='2' fill='%2300cfff'/%3E%3Ccircle cx='90' cy='50' r='2' fill='%2300cfff'/%3E%3C/svg%3E");
    background-size: 100px 100px;
}

/* ── Hero header ── */
.hero {
    display: flex; align-items: center; gap: 1.5rem;
    padding: 2rem 0 1rem; border-bottom: 1px solid rgba(0,200,255,0.1);
    margin-bottom: 1.5rem; position: relative;
}
.hero-icon {
    width: 64px; height: 64px; flex-shrink: 0;
}
.hero-title {
    font-family: 'Rajdhani', sans-serif;
    font-size: 2.8rem; font-weight: 700; letter-spacing: 2px;
    color: #fff;
    text-shadow: 0 0 30px rgba(0,200,255,0.4), 0 0 60px rgba(0,200,255,0.15);
    line-height: 1;
}
.hero-title span { color: #00cfff; }
.hero-sub {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.78rem; color: #3a6080; letter-spacing: 1px;
    margin-top: 0.4rem;
}
.hero-badge {
    margin-left: auto; display: flex; flex-direction: column; gap: 4px; align-items: flex-end;
}
.badge-pill {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.68rem; letter-spacing: 1px;
    background: rgba(0,200,255,0.08); border: 1px solid rgba(0,200,255,0.2);
    color: #00cfff; padding: 3px 10px; border-radius: 3px;
}

/* ── Pipeline bar ── */
.pipeline {
    display: flex; align-items: center; gap: 0;
    margin-bottom: 1.5rem; overflow-x: auto;
    padding: 0.75rem 1rem;
    background: rgba(0,200,255,0.03);
    border: 1px solid rgba(0,200,255,0.08);
    border-radius: 6px;
}
.pipe-step {
    display: flex; align-items: center; gap: 0.4rem;
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.7rem; color: #3a6080; white-space: nowrap;
    padding: 0 0.5rem;
}
.pipe-step.active { color: #00cfff; }
.pipe-step .dot {
    width: 6px; height: 6px; border-radius: 50%;
    background: #1a2a35; border: 1px solid #3a6080; flex-shrink: 0;
}
.pipe-step.active .dot { background: #00cfff; box-shadow: 0 0 6px #00cfff; }
.pipe-arrow { color: #1a3040; font-size: 0.8rem; padding: 0 2px; flex-shrink: 0; }

/* ── Search bar ── */
.search-wrap {
    background: rgba(0,200,255,0.04);
    border: 1px solid rgba(0,200,255,0.15);
    border-radius: 8px; padding: 1.2rem 1.4rem;
    margin-bottom: 1rem;
    box-shadow: 0 0 40px rgba(0,200,255,0.04) inset;
}

.stTextInput > div > div > input {
    background: rgba(0,0,0,0.4) !important;
    color: #c8e8ff !important;
    border: 1px solid rgba(0,200,255,0.2) !important;
    border-radius: 6px !important;
    font-family: 'Exo 2', sans-serif !important;
    font-size: 1rem !important;
    padding: 0.7rem 1rem !important;
    letter-spacing: 0.3px;
    transition: border-color 0.2s, box-shadow 0.2s;
}
.stTextInput > div > div > input:focus {
    border-color: rgba(0,200,255,0.5) !important;
    box-shadow: 0 0 0 3px rgba(0,200,255,0.1) !important;
    outline: none !important;
}
.stTextInput > div > div > input::placeholder { color: #2a4a60 !important; }

/* ── Buttons ── */
.stButton > button[kind="primary"] {
    background: linear-gradient(135deg, #00cfff 0%, #0088bb 100%) !important;
    color: #020810 !important; border: none !important;
    font-family: 'Rajdhani', sans-serif !important;
    font-weight: 700 !important; font-size: 1rem !important;
    letter-spacing: 1.5px !important;
    border-radius: 6px !important;
    box-shadow: 0 0 20px rgba(0,200,255,0.25) !important;
    transition: all 0.2s !important;
}
.stButton > button[kind="primary"]:hover {
    box-shadow: 0 0 35px rgba(0,200,255,0.45) !important;
    transform: translateY(-1px) !important;
}

/* ── Chips ── */
.chip-row { display: flex; flex-wrap: wrap; gap: 6px; margin: 0.5rem 0 1rem; }
.chip {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.72rem; letter-spacing: 0.5px;
    background: rgba(0,200,255,0.05);
    border: 1px solid rgba(0,200,255,0.15);
    color: #4a8aa8; border-radius: 4px;
    padding: 4px 12px; cursor: default;
    transition: all 0.15s;
}
.chip:hover { border-color: rgba(0,200,255,0.4); color: #00cfff; }

/* ── Status bar ── */
.status-bar {
    display: flex; gap: 1.5rem; align-items: center;
    font-family: 'Share Tech Mono', monospace; font-size: 0.75rem;
    color: #2a5060; margin-bottom: 1rem;
    padding-bottom: 0.75rem;
    border-bottom: 1px solid rgba(0,200,255,0.06);
}
.status-item { display: flex; gap: 0.4rem; align-items: center; }
.status-dot { width: 5px; height: 5px; border-radius: 50%; background: #00cfff;
              box-shadow: 0 0 4px #00cfff; animation: blink 2s infinite; }
@keyframes blink { 0%,100%{opacity:1} 50%{opacity:0.3} }

/* ── Answer box ── */
.answer-panel {
    background: rgba(0,12,25,0.8);
    border: 1px solid rgba(0,200,255,0.2);
    border-radius: 8px; overflow: hidden;
    margin: 0.75rem 0 1.5rem;
    box-shadow: 0 0 60px rgba(0,200,255,0.05);
}
.answer-header {
    background: rgba(0,200,255,0.07);
    border-bottom: 1px solid rgba(0,200,255,0.15);
    padding: 0.6rem 1.2rem;
    display: flex; align-items: center; gap: 0.75rem;
}
.answer-header-label {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.7rem; letter-spacing: 2px; color: #00cfff; text-transform: uppercase;
}
.latency-tag {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.68rem; color: #00ff88;
    background: rgba(0,255,136,0.08); border: 1px solid rgba(0,255,136,0.2);
    padding: 2px 8px; border-radius: 3px; margin-left: auto;
}
.answer-body {
    padding: 1.4rem 1.6rem;
    font-family: 'Exo 2', sans-serif;
    font-size: 0.97rem; line-height: 1.9; color: #b8d8f0;
    white-space: pre-wrap;
}
.answer-body strong { color: #00cfff; }

/* ── Abstention ── */
.abstain-panel {
    background: rgba(20,12,0,0.8);
    border: 1px solid rgba(255,160,0,0.25);
    border-radius: 8px; padding: 1.2rem 1.5rem;
    margin: 0.75rem 0 1.5rem;
}
.abstain-label {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.68rem; letter-spacing: 2px; color: #ff9900;
    text-transform: uppercase; margin-bottom: 0.5rem;
}
.abstain-text { font-size: 0.93rem; color: #c0982a; line-height: 1.7; }

/* ── References ── */
.ref-section-label {
    font-family: 'Share Tech Mono', monospace;
    font-size: 0.68rem; letter-spacing: 2.5px; color: #2a5060;
    text-transform: uppercase; margin: 1.5rem 0 0.75rem;
    display: flex; align-items: center; gap: 0.75rem;
}
.ref-section-label::after {
    content: ''; flex: 1; height: 1px;
    background: linear-gradient(90deg, rgba(0,200,255,0.15), transparent);
}
.ref-card {
    background: rgba(0,8,18,0.7);
    border: 1px solid rgba(0,200,255,0.08);
    border-left: 3px solid rgba(0,200,255,0.3);
    border-radius: 6px; padding: 0.9rem 1.1rem;
    margin-bottom: 0.6rem;
    transition: border-color 0.2s;
}
.ref-card:hover { border-color: rgba(0,200,255,0.25); border-left-color: #00cfff; }
.ref-top { display: flex; align-items: flex-start; gap: 0.75rem; margin-bottom: 0.4rem; }
.ref-num {
    font-family: 'Share Tech Mono', monospace; font-size: 0.72rem;
    color: #00cfff; background: rgba(0,200,255,0.08);
    border: 1px solid rgba(0,200,255,0.2);
    padding: 1px 8px; border-radius: 3px; flex-shrink: 0; margin-top: 1px;
}
.ref-title { font-family: 'Rajdhani', sans-serif; font-size: 1rem; font-weight: 600; color: #8ab8d8; }
.ref-score {
    font-family: 'Share Tech Mono', monospace; font-size: 0.65rem;
    color: #2a7090; margin-left: auto; flex-shrink: 0;
    background: rgba(0,200,255,0.05); border: 1px solid rgba(0,200,255,0.1);
    padding: 2px 8px; border-radius: 3px;
}
.ref-url { font-size: 0.78rem; color: #1a6080; margin: 0.3rem 0; word-break: break-all; }
.ref-url a { color: #2a8aaa; text-decoration: none; }
.ref-url a:hover { color: #00cfff; }
.ref-text { font-size: 0.84rem; color: #3a5870; line-height: 1.65; margin-top: 0.4rem; }

/* ── Empty state ── */
.empty-state {
    text-align: center; padding: 4rem 2rem;
    display: flex; flex-direction: column; align-items: center; gap: 1rem;
}
.empty-glyph { opacity: 0.12; }
.empty-title {
    font-family: 'Rajdhani', sans-serif; font-size: 1.3rem;
    font-weight: 600; color: #2a4060; letter-spacing: 1px;
}
.empty-sub { font-family: 'Share Tech Mono', monospace; font-size: 0.72rem; color: #1a3040; }

/* ── Sidebar ── */
[data-testid="stSidebar"] {
    background: #030610 !important;
    border-right: 1px solid rgba(0,200,255,0.08) !important;
}
[data-testid="stSidebar"] .block-container { padding: 1.5rem 1rem; }

.sidebar-logo {
    font-family: 'Rajdhani', sans-serif; font-size: 1.3rem; font-weight: 700;
    color: #00cfff; letter-spacing: 2px; margin-bottom: 0.2rem;
}
.sidebar-sub {
    font-family: 'Share Tech Mono', monospace; font-size: 0.65rem;
    color: #1a4050; letter-spacing: 1px; margin-bottom: 1rem;
}
.sidebar-divider { border: none; border-top: 1px solid rgba(0,200,255,0.08); margin: 0.75rem 0; }
.sidebar-label {
    font-family: 'Share Tech Mono', monospace; font-size: 0.65rem;
    color: #1a4050; letter-spacing: 2px; text-transform: uppercase; margin-bottom: 0.4rem;
}
.stat-row { display: flex; justify-content: space-between; align-items: center; margin-bottom: 0.3rem; }
.stat-key { font-family: 'Share Tech Mono', monospace; font-size: 0.7rem; color: #2a5060; }
.stat-val { font-family: 'Share Tech Mono', monospace; font-size: 0.7rem; color: #00cfff; }

.pipe-list { display: flex; flex-direction: column; gap: 4px; margin-bottom: 0.5rem; }
.pipe-item {
    display: flex; align-items: center; gap: 0.5rem;
    font-family: 'Share Tech Mono', monospace; font-size: 0.68rem; color: #2a5060;
}
.pipe-item::before { content: '\u25b6'; font-size: 0.5rem; color: rgba(0,200,255,0.4); }

.demo-q {
    font-family: 'Exo 2', sans-serif; font-size: 0.78rem; color: #2a4860;
    padding: 4px 0; border-bottom: 1px solid rgba(0,200,255,0.05);
}

/* ── Slider ── */
.stSlider [data-baseweb="slider"] { padding: 0 4px; }
.stSlider [data-baseweb="slider"] > div > div { background: rgba(0,200,255,0.15) !important; }
.stSlider [data-baseweb="slider"] [role="slider"] { background: #00cfff !important; box-shadow: 0 0 8px #00cfff !important; }

/* ── Status widget ── */
[data-testid="stStatusWidget"] {
    background: rgba(0,12,25,0.9) !important;
    border: 1px solid rgba(0,200,255,0.2) !important;
    border-radius: 8px !important;
    font-family: 'Share Tech Mono', monospace !important;
}

/* ── Warning ── */
.stAlert { background: rgba(20,12,0,0.6) !important; border-color: rgba(255,160,0,0.3) !important; }
</style>
"""

st.markdown(CSS, unsafe_allow_html=True)

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT      = Path("/content/drive/MyDrive/ifixit_dump/final_hybrid_rag")
IDX_PATH  = ROOT / "index.faiss"
META_PATH = ROOT / "meta.jsonl"
BM25_PATH = ROOT / "bm25.pkl"

DEMOS = [
    "How do I replace a MacBook Air battery?",
    "How do I replace an iPhone screen?",
    "How do I fix a laptop keyboard?",
    "How do I replace a phone charging port?",
    "How do I open a Nintendo Switch Joy-Con?",
]

PIPELINE_STEPS = ["BM25 SPARSE", "BGE DENSE", "RRF FUSION", "CROSS-ENCODER", "GPT-4o"]

# ── Loaders ────────────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def load_all():
    t0 = time.time()
    bi  = SentenceTransformer("BAAI/bge-small-en-v1.5")
    ce  = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    idx = faiss.read_index(str(IDX_PATH))
    with open(BM25_PATH, "rb") as f:
        bm25 = pickle.load(f)
    meta = []
    with open(META_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line: meta.append(json.loads(line))
    return dict(bi=bi, ce=ce, index=idx, bm25=bm25, meta=meta,
                load_time=round(time.time()-t0, 1))

# ── Helpers ────────────────────────────────────────────────────────────────────
def _get(item, *keys, default=""):
    for k in keys:
        if item.get(k): return str(item[k])
    return default

get_text  = lambda i: _get(i, "text","chunk","content","body")
get_title = lambda i: _get(i, "title","guide_title","name", default="Untitled guide")
get_url   = lambda i: _get(i, "url","guide_url","source_url")

def dense_retrieve(query, bi, index, meta, top_k=50):
    q = bi.encode(
        ["Represent this sentence for searching relevant passages: " + query],
        convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q, top_k)
    return {int(i): float(s) for s, i in zip(scores[0], ids[0]) if i >= 0}

def sparse_retrieve(query, bm25, top_k=50):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_ids = np.argsort(scores)[::-1][:top_k]
    return {int(i): float(scores[i]) for i in top_ids if scores[i] > 0}

def rrf(dense_scores, sparse_scores, k=60):
    all_ids = set(dense_scores) | set(sparse_scores)
    dr = {i: r+1 for r, i in enumerate(sorted(dense_scores, key=dense_scores.get, reverse=True))}
    sr = {i: r+1 for r, i in enumerate(sorted(sparse_scores, key=sparse_scores.get, reverse=True))}
    fused = {i: 1/(k+dr.get(i,1000)) + 1/(k+sr.get(i,1000)) for i in all_ids}
    return sorted(fused, key=fused.get, reverse=True)

def rerank(query, candidate_ids, meta, ce, top_n=5):
    pairs  = [(query, get_text(meta[i])) for i in candidate_ids if i < len(meta)]
    scores = ce.predict(pairs)
    ranked = sorted(zip(candidate_ids, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

def generate_answer(query, chunks_meta, api_key):
    client  = OpenAI(api_key=api_key)
    context = "\n\n".join(
        f"[{i+1}] {get_title(m)}\n{get_text(m)}"
        for i, (_, m) in enumerate(chunks_meta)
    )
    system = (
        "You are an expert device repair assistant. "
        "Answer using ONLY the provided guide excerpts. "
        "Format your answer as clear numbered steps. "
        "Cite sources inline as [1], [2] etc. "
        "If evidence is insufficient, respond with exactly: INSUFFICIENT_EVIDENCE"
    )
    resp = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role":"system","content":system},
            {"role":"user","content":f"Question: {query}\n\nGuide excerpts:\n{context}"}
        ],
        max_tokens=650, temperature=0.2,
    )
    return resp.choices[0].message.content.strip()

# ── Sidebar ────────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown('<div class="sidebar-logo">\u26a1 REPAIRRAG</div>', unsafe_allow_html=True)
    st.markdown('<div class="sidebar-sub">HYBRID RAG SYSTEM v2.0</div>', unsafe_allow_html=True)
    st.markdown('<hr class="sidebar-divider">', unsafe_allow_html=True)

    with st.spinner("Initializing systems…"):
        try:
            R = load_all(); ready = True
        except Exception as e:
            ready = False; load_err = str(e)

    if ready:
        st.markdown(
            '<div style="display:flex;align-items:center;gap:6px;margin-bottom:0.75rem;">'
            '<div style="width:7px;height:7px;border-radius:50%;background:#00ff88;box-shadow:0 0 6px #00ff88;animation:blink 2s infinite;"></div>'
            '<span style="font-family:Share Tech Mono,monospace;font-size:0.7rem;color:#00ff88;letter-spacing:1px;">SYSTEMS ONLINE</span>'
            '</div>', unsafe_allow_html=True)
        st.markdown(
            f'<div class="stat-row"><span class="stat-key">CHUNKS</span><span class="stat-val">{len(R["meta"]):,}</span></div>'
            f'<div class="stat-row"><span class="stat-key">LOAD TIME</span><span class="stat-val">{R["load_time"]}s</span></div>'
            f'<div class="stat-row"><span class="stat-key">ENCODER</span><span class="stat-val">BGE-SMALL</span></div>'
            f'<div class="stat-row"><span class="stat-key">RERANKER</span><span class="stat-val">MS-MARCO</span></div>',
            unsafe_allow_html=True)
    else:
        st.markdown('<span style="color:#ff4040;font-family:Share Tech Mono,monospace;font-size:0.75rem;">LOAD FAILED</span>', unsafe_allow_html=True)
        st.code(load_err); st.stop()

    st.markdown('<hr class="sidebar-divider">', unsafe_allow_html=True)
    st.markdown('<div class="sidebar-label">OpenAI API Key</div>', unsafe_allow_html=True)
    api_key = st.text_input("key", type="password", placeholder="sk-...", label_visibility="collapsed")

    st.markdown('<hr class="sidebar-divider">', unsafe_allow_html=True)
    st.markdown('<div class="sidebar-label">Retrieved Chunks</div>', unsafe_allow_html=True)
    top_k = st.slider("k", 1, 10, 5, label_visibility="collapsed")

    st.markdown('<hr class="sidebar-divider">', unsafe_allow_html=True)
    st.markdown('<div class="sidebar-label">Pipeline</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="pipe-list">'
        '<div class="pipe-item">BM25 Sparse Retrieval</div>'
        '<div class="pipe-item">BGE Dense Retrieval</div>'
        '<div class="pipe-item">Reciprocal Rank Fusion</div>'
        '<div class="pipe-item">Cross-Encoder Reranking</div>'
        '<div class="pipe-item">GPT-4o Grounded Answer</div>'
        '<div class="pipe-item">Abstention Guard</div>'
        '</div>', unsafe_allow_html=True)

    st.markdown('<hr class="sidebar-divider">', unsafe_allow_html=True)
    st.markdown('<div class="sidebar-label">Example Queries</div>', unsafe_allow_html=True)
    for q in DEMOS:
        st.markdown(f'<div class="demo-q">{q}</div>', unsafe_allow_html=True)

# ── Hero ───────────────────────────────────────────────────────────────────────
st.markdown("""
<div class="hero">
  <svg class="hero-icon" viewBox="0 0 64 64" fill="none" xmlns="http://www.w3.org/2000/svg">
    <rect x="2" y="10" width="60" height="44" rx="4" fill="#050d1a" stroke="#00cfff" stroke-width="1.5"/>
    <rect x="8" y="16" width="16" height="10" rx="2" fill="none" stroke="#00cfff" stroke-width="1" opacity="0.5"/>
    <rect x="28" y="16" width="8" height="8" rx="1" fill="none" stroke="#00aadd" stroke-width="1" opacity="0.4"/>
    <rect x="40" y="16" width="14" height="6" rx="1" fill="none" stroke="#00aadd" stroke-width="1" opacity="0.3"/>
    <line x1="8" y1="32" x2="56" y2="32" stroke="#00cfff" stroke-width="0.5" opacity="0.2"/>
    <circle cx="12" cy="38" r="3" fill="none" stroke="#00cfff" stroke-width="1" opacity="0.6"/>
    <circle cx="22" cy="38" r="3" fill="none" stroke="#00cfff" stroke-width="1" opacity="0.4"/>
    <rect x="30" y="35" width="10" height="6" rx="1" fill="#00cfff" opacity="0.15" stroke="#00cfff" stroke-width="0.8"/>
    <rect x="44" y="35" width="6" height="3" rx="0.5" fill="none" stroke="#00aadd" stroke-width="0.8" opacity="0.4"/>
    <rect x="44" y="40" width="6" height="3" rx="0.5" fill="none" stroke="#00aadd" stroke-width="0.8" opacity="0.4"/>
    <line x1="2" y1="48" x2="62" y2="48" stroke="#00cfff" stroke-width="0.5" opacity="0.15"/>
    <rect x="8" y="50" width="48" height="2" rx="1" fill="#00cfff" opacity="0.08"/>
    <line x1="20" y1="54" x2="20" y2="58" stroke="#00cfff" stroke-width="2" stroke-linecap="round" opacity="0.5"/>
    <line x1="32" y1="54" x2="32" y2="58" stroke="#00cfff" stroke-width="2" stroke-linecap="round" opacity="0.3"/>
    <line x1="44" y1="54" x2="44" y2="58" stroke="#00cfff" stroke-width="2" stroke-linecap="round" opacity="0.5"/>
  </svg>
  <div>
    <div class="hero-title">Repair<span>RAG</span></div>
    <div class="hero-sub">BM25 + BGE DENSE &nbsp;&#x2192;&nbsp; RRF FUSION &nbsp;&#x2192;&nbsp; CROSS-ENCODER &nbsp;&#x2192;&nbsp; GPT-4o</div>
  </div>
  <div class="hero-badge">
    <div class="badge-pill">iFixit Corpus</div>
    <div class="badge-pill">165K+ Chunks</div>
    <div class="badge-pill">Hybrid RAG v2</div>
  </div>
</div>
""", unsafe_allow_html=True)

# ── Pipeline indicator ─────────────────────────────────────────────────────────
pipe_html = '<div class="pipeline">'
for i, step in enumerate(PIPELINE_STEPS):
    pipe_html += f'<div class="pipe-step"><div class="dot"></div>{step}</div>'
    if i < len(PIPELINE_STEPS)-1:
        pipe_html += '<div class="pipe-arrow">&#x25B6;</div>'
pipe_html += '</div>'
st.markdown(pipe_html, unsafe_allow_html=True)

# ── Search ─────────────────────────────────────────────────────────────────────
st.markdown('<div class="search-wrap">', unsafe_allow_html=True)
col_q, col_btn = st.columns([5.5, 1])
with col_q:
    query = st.text_input("q", placeholder="e.g.  How do I replace a MacBook Air battery?",
                          label_visibility="collapsed")
with col_btn:
    ask = st.button("SEARCH", type="primary", use_container_width=True)
st.markdown('</div>', unsafe_allow_html=True)

chips_html = '<div class="chip-row">' + "".join(f'<span class="chip">{q}</span>' for q in DEMOS) + '</div>'
st.markdown(chips_html, unsafe_allow_html=True)

# ── Results ────────────────────────────────────────────────────────────────────
if ask and query.strip():
    if not api_key.strip():
        st.warning("\u26a0 Please enter your OpenAI API key in the sidebar to generate answers.")
        st.stop()

    with st.status("\u26a1 Running hybrid retrieval pipeline…", expanded=True) as status:
        t0 = time.time()
        st.write("[ BM25 ] Sparse retrieval…")
        sparse = sparse_retrieve(query, R["bm25"])
        st.write("[ BGE ] Dense retrieval…")
        dense  = dense_retrieve(query, R["bi"], R["index"], R["meta"])
        st.write("[ RRF ] Fusing rankings…")
        fused  = rrf(dense, sparse)[:100]
        st.write("[ CE ] Cross-encoder reranking…")
        reranked = rerank(query, fused, R["meta"], R["ce"], top_n=top_k)
        chunks_meta = [(score, R["meta"][i]) for i, score in reranked if i < len(R["meta"])]
        st.write("[ GPT-4o ] Generating grounded answer…")
        try:
            answer = generate_answer(query, chunks_meta, api_key)
        except Exception as e:
            answer = f"Generation error: {e}"
        latency = round(time.time()-t0, 2)
        status.update(label=f"\u2713 Pipeline complete in {latency}s", state="complete")

    # ── Answer panel ───────────────────────────────────────────────────────────
    if "INSUFFICIENT_EVIDENCE" in answer:
        st.markdown(
            '<div class="abstain-panel">'
            '<div class="abstain-label">\u26a0 Abstention Triggered</div>'
            '<div class="abstain-text">The retrieved guides do not contain sufficient evidence to answer this question confidently. '
            'Try rephrasing your question or specifying a device model and year.</div>'
            '</div>', unsafe_allow_html=True)
    else:
        # Format citations bold
        formatted = re.sub(r'\[(\d+)\]', r'<strong>[\1]</strong>', answer)
        st.markdown(
            f'<div class="answer-panel">'
            f'<div class="answer-header">'
            f'<svg width="14" height="14" viewBox="0 0 14 14" fill="none"><circle cx="7" cy="7" r="6" stroke="#00cfff" stroke-width="1.2"/><path d="M7 4v4M7 9.5v.5" stroke="#00cfff" stroke-width="1.2" stroke-linecap="round"/></svg>'
            f'<span class="answer-header-label">Generated Answer</span>'
            f'<span class="latency-tag">\u26a1 {latency}s</span>'
            f'</div>'
            f'<div class="answer-body">{formatted}</div>'
            f'</div>', unsafe_allow_html=True)

    # ── References ─────────────────────────────────────────────────────────────
    st.markdown(
        '<div class="ref-section-label">'
        '<svg width="12" height="12" viewBox="0 0 12 12" fill="none"><rect x="1" y="1" width="10" height="10" rx="2" stroke="#2a5060" stroke-width="1"/><line x1="3" y1="4" x2="9" y2="4" stroke="#2a5060" stroke-width="1"/><line x1="3" y1="6" x2="9" y2="6" stroke="#2a5060" stroke-width="1"/><line x1="3" y1="8" x2="7" y2="8" stroke="#2a5060" stroke-width="1"/></svg>'
        f'References &mdash; top {len(chunks_meta)} sources'
        '</div>', unsafe_allow_html=True)

    for rank, (ce_score, item) in enumerate(chunks_meta, 1):
        title = get_title(item)
        text  = get_text(item)
        url   = get_url(item)
        snip  = " ".join(text.split())[:280] + ("\u2026" if len(text) > 280 else "")
        url_html = f'<div class="ref-url"><a href="{url}" target="_blank">{url}</a></div>' if url else ""
        st.markdown(
            f'<div class="ref-card">'
            f'<div class="ref-top">'
            f'<span class="ref-num">#{rank}</span>'
            f'<span class="ref-title">{title}</span>'
            f'<span class="ref-score">score {ce_score:.3f}</span>'
            f'</div>'
            f'{url_html}'
            f'<div class="ref-text">{snip}</div>'
            f'</div>', unsafe_allow_html=True)

elif ask:
    st.warning("Please enter a repair question first.")

else:
    st.markdown("""
<div class="empty-state">
  <svg class="empty-glyph" width="180" height="140" viewBox="0 0 180 140" fill="none" xmlns="http://www.w3.org/2000/svg">
    <rect x="10" y="20" width="160" height="100" rx="6" fill="none" stroke="#00cfff" stroke-width="1.5"/>
    <rect x="20" y="30" width="50" height="30" rx="3" fill="none" stroke="#00cfff" stroke-width="1"/>
    <rect x="80" y="30" width="30" height="12" rx="2" fill="none" stroke="#00aadd" stroke-width="1"/>
    <rect x="80" y="46" width="30" height="12" rx="2" fill="none" stroke="#00aadd" stroke-width="1"/>
    <rect x="118" y="30" width="40" height="28" rx="3" fill="none" stroke="#00aadd" stroke-width="1"/>
    <line x1="10" y1="72" x2="170" y2="72" stroke="#00cfff" stroke-width="0.5"/>
    <circle cx="30" cy="88" r="8" fill="none" stroke="#00cfff" stroke-width="1"/>
    <circle cx="55" cy="88" r="8" fill="none" stroke="#00cfff" stroke-width="0.7"/>
    <rect x="72" y="82" width="36" height="12" rx="2" fill="none" stroke="#00cfff" stroke-width="1"/>
    <rect x="116" y="82" width="20" height="5" rx="1" fill="none" stroke="#00aadd" stroke-width="0.8"/>
    <rect x="116" y="90" width="20" height="5" rx="1" fill="none" stroke="#00aadd" stroke-width="0.8"/>
    <rect x="142" y="82" width="16" height="13" rx="2" fill="#00cfff" opacity="0.08" stroke="#00cfff" stroke-width="0.8"/>
    <line x1="10" y1="105" x2="170" y2="105" stroke="#00cfff" stroke-width="0.5" opacity="0.4"/>
    <line x1="40" y1="105" x2="40" y2="120" stroke="#00cfff" stroke-width="2" stroke-linecap="round" opacity="0.5"/>
    <line x1="90" y1="105" x2="90" y2="120" stroke="#00cfff" stroke-width="2" stroke-linecap="round" opacity="0.3"/>
    <line x1="140" y1="105" x2="140" y2="120" stroke="#00cfff" stroke-width="2" stroke-linecap="round" opacity="0.5"/>
    <circle cx="40" cy="125" r="4" fill="none" stroke="#00cfff" stroke-width="1" opacity="0.5"/>
    <circle cx="90" cy="125" r="4" fill="none" stroke="#00cfff" stroke-width="1" opacity="0.3"/>
    <circle cx="140" cy="125" r="4" fill="none" stroke="#00cfff" stroke-width="1" opacity="0.5"/>
  </svg>
  <div class="empty-title">AWAITING QUERY INPUT</div>
  <div class="empty-sub">TYPE A REPAIR QUESTION ABOVE &nbsp;&#x25B6;&nbsp; CLICK SEARCH</div>
</div>
""", unsafe_allow_html=True)


Overwriting /content/app.py


In [42]:
# 5. Launch Streamlit
import subprocess, time

subprocess.run('pkill -f streamlit', shell=True, check=False)
time.sleep(2)

log = open('/content/streamlit.log', 'w')
subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=log, stderr=log
)
time.sleep(10)
print('✓ Streamlit running on port 8501')

✓ Streamlit running on port 8501


In [43]:
import re
from IPython.display import HTML, display

log_text = open('/content/ngrok.log').read()
match = re.search(r'https://[a-zA-Z0-9-]+\.ngrok-free\.[a-z]+', log_text)

if match:
    url = match.group(0)
    print(f'Live at: {url}')
    display(HTML(f'''
    <div style="font-family:monospace;background:#050810;border:1px solid rgba(0,200,255,0.3);
                border-radius:8px;padding:1.5rem 2rem;margin:1rem 0;">
      <div style="color:#00cfff;font-size:0.7rem;letter-spacing:2px;margin-bottom:0.75rem;">⚡ REPAIRRAG IS LIVE</div>
      <a href="{url}" target="_blank"
         style="background:linear-gradient(135deg,#00cfff,#0088bb);color:#020810;
                padding:10px 28px;border-radius:6px;text-decoration:none;font-weight:700;font-size:0.95rem;">
        🔗 OPEN REPAIRRAG DASHBOARD
      </a>
    </div>
    '''))
else:
    print('Still not found, raw log:')
    print(log_text[-500:])


Live at: https://agile-canon-nutlike.ngrok-free.dev


In [44]:
import pickle
bm25 = pickle.load(open('/content/drive/MyDrive/ifixit_dump/final_hybrid_rag/bm25.pkl', 'rb'))
print(type(bm25))
print(bm25.keys() if isinstance(bm25, dict) else dir(bm25))

<class 'dict'>
dict_keys(['tokenized_corpus'])


In [45]:
# (Optional) View Streamlit logs
!tail -40 /content/streamlit.log




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.67.3.248:8501



## Troubleshooting

| Problem | Fix |
|---|---|
| `LOAD FAILED` in sidebar | Verify all 3 files exist in `MyDrive/ifixit_dump/final_hybrid_rag/` |
| No ngrok URL | Run `!cat /content/ngrok.log` to see the error |
| `ERR_NGROK_108` | Run `!pkill -f ngrok` then re-run cell 6 |
| Answer box empty | Paste OpenAI API key in the sidebar |
| `INSUFFICIENT_EVIDENCE` | Try a more specific device repair question |
| Slow startup (2–3 min) | Normal — loading 400MB+ indexes takes time |

In [46]:
import re

app = open('/content/app.py').read()

old = '    with open(BM25_PATH, "rb") as f:\n        bm25 = pickle.load(f)'

new = """    with open(BM25_PATH, "rb") as f:
        bm25_data = pickle.load(f)
    from rank_bm25 import BM25Okapi
    bm25 = BM25Okapi(bm25_data["tokenized_corpus"])"""

app = app.replace(old, new)
open('/content/app.py', 'w').write(app)
print("Patched!")

Patched!


In [47]:
import subprocess, time
subprocess.run('pkill -f streamlit', shell=True, check=False)
time.sleep(2)
log = open('/content/streamlit.log', 'w')
subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=log, stderr=log
)
time.sleep(8)
print("Done — reload your ngrok URL!")

Done — reload your ngrok URL!


In [48]:
import faiss
from sentence_transformers import SentenceTransformer

index = faiss.read_index('/content/drive/MyDrive/ifixit_dump/final_hybrid_rag/index.faiss')
print("Index dimension:", index.d)

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
test = model.encode(["test"], convert_to_numpy=True)
print("Model dimension:", test.shape[1])

Index dimension: 1024


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model dimension: 384


In [49]:
import re

app = open('/content/app.py').read()
app = app.replace(
    'SentenceTransformer("BAAI/bge-small-en-v1.5")',
    'SentenceTransformer("BAAI/bge-large-en-v1.5")'
)
open('/content/app.py', 'w').write(app)
print("Patched! Now restart Streamlit.")

Patched! Now restart Streamlit.


In [50]:
import subprocess, time
subprocess.run('pkill -f streamlit', shell=True, check=False)
time.sleep(2)
log = open('/content/streamlit.log', 'w')
subprocess.Popen(
    ['streamlit', 'run', '/content/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=log, stderr=log
)
time.sleep(10)
print("Done — reload your ngrok URL!")

Done — reload your ngrok URL!
